[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 02](README.md)

# Pthreads: ciclo de vida y partición

**Tema:** 02 · **Sesiones:** 7 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo crear trabajo concurrente sin perder argumentos, errores ni cobertura de datos?


## Resultados de aprendizaje

- Explicar create/join y la vida útil de argumentos.
- Particionar datos con cobertura comprobable.
- Comparar la salida paralela con una referencia serial.


## Modelo conceptual

`pthread_create` inicia una función con un argumento cuya vida útil debe abarcar el acceso del hilo.

`pthread_join` establece finalización y permite recuperar estado; ignorar códigos de retorno oculta fallos.

La partición debe especificar rangos semiabiertos y funcionar cuando n no es múltiplo del número de hilos.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "02"
NOTEBOOK = "02_memoria_compartida/01_pthreads.ipynb"
assert (ROOT / "curso" / "notebooks" / "02_memoria_compartida" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Rangos de trabajo

Se prueba una partición por bloques para casos irregulares.


In [ ]:
def partition(n, workers):
    q, r = divmod(n, workers)
    starts = [worker * q + min(worker, r) for worker in range(workers)]
    return [(start, start + q + (worker < r)) for worker, start in enumerate(starts)]
for n, workers in ((3, 5), (17, 4), (32, 8)):
    chunks = partition(n, workers)
    flattened = [i for begin, end in chunks for i in range(begin, end)]
    assert flattened == list(range(n))
    print(n, workers, chunks)


**Interpretación.** Los hilos sin elementos reciben un rango vacío válido; el programa no debe leer fuera de límites.


## Referencia y reducción

Se simula la suma de parciales y se compara con una referencia única.


In [ ]:
values = [((i * 17) % 23) - 11 for i in range(101)]
chunks = partition(len(values), 6)
partials = [sum(values[begin:end]) for begin, end in chunks]
parallel_result = sum(partials)
reference = sum(values)
assert parallel_result == reference
print({"partials": partials, "result": parallel_result})


**Interpretación.** En C, cada parcial debe tener almacenamiento independiente y la combinación ocurre después de join.


## Práctica reproducible

1. Compilar y revisar `pthreads/thread_creation.c`.
2. Agregar comprobación de cada retorno de la API.
3. Probar n<threads, n no divisible y n grande.


## Errores frecuentes

- Pasar la dirección de una variable de bucle compartida.
- Salir de `main` antes de join.
- Medir una versión paralela incorrecta.

## Criterios de aceptación

- Todos los retornos se comprueban.
- Cobertura de índices demostrada.
- Resultado igual a la referencia serial.


## Referencias y material relacionado

- [Creación de hilos](../../../pthreads/thread_creation.c)
- [Ejemplo de mutex](../../../pthreads/thread_mutex.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 02](README.md)
